In [0]:
#|default_exp cleaning_functions_export

In [0]:
%pip install fastcore nbdev

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
#| export

from pyspark.sql.window import Window
from pyspark.sql import SparkSession, DataFrame, functions as F
from delta.tables import DeltaTable
from datetime import datetime, UTC

In [0]:
import nbdev
from fastcore.test import *
from pyspark.sql import Row
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType


def _make_sg(**overrides):
    """Build a single stopGroup Row with sensible defaults."""
    defaults = dict(node=1, cis=100, name="A", fullName="A full",
                    uniqueName="A_u", municipality="Prague", districtCode="P1",
                    avgLat="50.0", avgLon="14.0", avgJtskX="1", avgJtskY="2",
                    isTrain="false", mainTrafficType="bus", idosCategory="1",
                    idosName="A_idos", stops=[Row(id="s1")])
    return Row(**{**defaults, **overrides})


def _make_bronze(stop_groups, generated_at="2026-04-18T12:00:00"):
    """Build a minimal bronze-shaped DataFrame."""
    return spark.createDataFrame([Row(stopGroups=stop_groups, generatedAt=generated_at)])

In [0]:
#| export

def explode_stop_groups(bronze_stops_df: DataFrame) -> DataFrame:
    """Flatten stopGroups array into one row per stop-group with source_timestamp."""
    return (
        bronze_stops_df
        .select(
            F.explode("stopGroups").alias("sg"),
            F.to_timestamp("generatedAt").alias("source_timestamp"),
        )
        .select("sg.*", "source_timestamp")
    )

In [0]:
bronze = _make_bronze([_make_sg(node=1, cis=100), _make_sg(node=2, cis=200, name="C")])
result = explode_stop_groups(bronze)

# Flattens array into individual rows
test_eq(result.count(), 2)
test_eq(set(result.select("node").toPandas()["node"]), {1, 2})

# source_timestamp is a proper TimestampType
ts_type = [f.dataType for f in result.schema.fields if f.name == "source_timestamp"][0]
test_eq(type(ts_type), type(TimestampType()))

print("All explode_stop_groups tests passed!")

All explode_stop_groups tests passed!


In [0]:
#| export

def deduplicate_stops(stops_df: DataFrame) -> DataFrame:
    """Keep only the latest row per (node, cis) based on source_timestamp."""
    w = Window.partitionBy("node", "cis").orderBy(F.col("source_timestamp").desc())
    return stops_df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

In [0]:
df_old = explode_stop_groups(_make_bronze([_make_sg(name="Old")], "2026-04-17T10:00:00"))
df_new = explode_stop_groups(_make_bronze([_make_sg(name="New")], "2026-04-18T10:00:00"))

result = deduplicate_stops(df_old.union(df_new))

# Keeps only the latest row per (node, cis)
test_eq(result.count(), 1)
test_eq(result.collect()[0]["name"], "New")

print("All deduplicate_stops tests passed!")

All deduplicate_stops tests passed!


In [0]:
#| export

# Tracked attribute columns for change-detection hash
STOPS_TRACKED_COLS = [
    "name", "fullName", "uniqueName", "municipality", "districtCode",
    "avgLat", "avgLon", "avgJtskX", "avgJtskY",
    "isTrain", "mainTrafficType", "idosCategory", "idosName", "stops",
]


def compute_row_hash(df: DataFrame, cols: list[str] = STOPS_TRACKED_COLS) -> DataFrame:
    """Add MD5 hash over tracked attributes for change detection."""
    return df.withColumn(
        "row_hash",
        F.md5(F.concat_ws("||", *[F.to_json(F.struct(F.col(c))) for c in cols])),
    )

In [0]:
df = explode_stop_groups(_make_bronze([_make_sg()]))
result = compute_row_hash(df)

# row_hash column exists and is non-null
test_eq("row_hash" in result.columns, True)
test_eq(result.filter(col("row_hash").isNull()).count(), 0)

# Different data produces different hashes
result2 = compute_row_hash(explode_stop_groups(_make_bronze([_make_sg(name="X")])))
hash1 = result.collect()[0]["row_hash"]
hash2 = result2.collect()[0]["row_hash"]
assert hash1 != hash2, "Different data should produce different hashes"

print("All compute_row_hash tests passed!")

All compute_row_hash tests passed!


In [0]:
#| export

def prepare_stops_staging(bronze_stops_df: DataFrame) -> DataFrame:
    """Compose: explode → deduplicate → add row hash."""
    return compute_row_hash(deduplicate_stops(explode_stop_groups(bronze_stops_df)))

In [0]:
df_old = _make_bronze([_make_sg(name="A")], generated_at="2026-04-17T10:00:00")
df_new = _make_bronze([_make_sg(name="B")], generated_at="2026-04-18T10:00:00")
result = prepare_stops_staging(df_old.union(df_new))

# Dedup: one row per (node, cis), keeps latest
test_eq(result.count(), 1)
test_eq(result.collect()[0]["name"], "B")

# row_hash is present and non-null
test_eq(result.filter(col("row_hash").isNull()).count(), 0)

# Two distinct stop groups → two rows
result2 = prepare_stops_staging(_make_bronze([_make_sg(), _make_sg(node=2, cis=200, name="C")]))
test_eq(result2.count(), 2)

print("All prepare_stops_staging tests passed!")

All prepare_stops_staging tests passed!


In [0]:
#| export

def ensure_silver_stops_table(spark: SparkSession, silver_stops_table: str) -> None:
    """Create the silver SCD2 stops table if it does not already exist."""
    spark.sql(
        """
        CREATE TABLE IF NOT EXISTS IDENTIFIER(:silver_stops_table) (
          node BIGINT,
          cis BIGINT,
          name STRING,
          fullName STRING,
          uniqueName STRING,
          municipality STRING,
          districtCode STRING,
          avgLat DOUBLE,
          avgLon DOUBLE,
          avgJtskX DOUBLE,
          avgJtskY DOUBLE,
          isTrain BOOLEAN,
          mainTrafficType STRING,
          idosCategory BIGINT,
          idosName STRING,
          stops ARRAY<STRUCT<
            altIdosName: STRING,
            gtfsIds: ARRAY<STRING>,
            id: STRING,
            isMetro: BOOLEAN,
            jtskX: DOUBLE,
            jtskY: DOUBLE,
            lat: DOUBLE,
            lines: ARRAY<STRUCT<
              direction: STRING,
              direction2: STRING,
              exitOnly: BOOLEAN,
              id: BIGINT,
              isNight: BOOLEAN,
              name: STRING,
              type: STRING
            >>,
            lon: DOUBLE,
            mainTrafficType: STRING,
            platform: STRING,
            wheelchairAccess: STRING,
            zone: STRING
          >>,
          row_hash STRING,
          valid_from TIMESTAMP,
          valid_to TIMESTAMP,
          is_current BOOLEAN
        )
        """,
        args={"silver_stops_table": silver_stops_table},
    )

In [0]:
# ensure_silver_stops_table creates a Delta table via DDL.
# TODO: add integration test with a temporary table in your catalog.
print("ensure_silver_stops_table: DDL function — add integration test with real catalog")

ensure_silver_stops_table: DDL function — add integration test with real catalog


In [0]:
#| export

STOPS_SILVER_COLS = [
    "node", "cis", "name", "fullName", "uniqueName",
    "municipality", "districtCode",
    "avgLat", "avgLon", "avgJtskX", "avgJtskY",
    "isTrain", "mainTrafficType", "idosCategory", "idosName",
    "stops", "row_hash",
]


def find_stops_to_expire(staging_stops_df: DataFrame, silver_stops_df: DataFrame) -> DataFrame:
    """Return staging rows whose hash differs from the current silver version."""
    current = silver_stops_df.filter("is_current").select("node", "cis", "row_hash")
    return (
        staging_stops_df.alias("s")
        .join(current.alias("t"), ["node", "cis"])
        .filter("s.row_hash != t.row_hash")
        .select("s.*")
    )

In [0]:
staging = prepare_stops_staging(_make_bronze([_make_sg(name="Updated")]))

# Silver with a different hash → should find a record to expire
silver_df = spark.createDataFrame(
    [Row(node=1, cis=100, row_hash="old_hash", is_current=True)]
)
to_expire = find_stops_to_expire(staging, silver_df)
test_eq(to_expire.count(), 1)

# Same hash → nothing to expire
same_hash = staging.select("row_hash").collect()[0]["row_hash"]
silver_same = spark.createDataFrame(
    [Row(node=1, cis=100, row_hash=same_hash, is_current=True)]
)
test_eq(find_stops_to_expire(staging, silver_same).count(), 0)

print("All find_stops_to_expire tests passed!")

All find_stops_to_expire tests passed!


In [0]:
#| export

def build_new_stop_records(staging_stops_df: DataFrame, silver_stops_df: DataFrame) -> DataFrame:
    """Build rows to insert: genuinely new keys + just-expired changed keys."""
    current_keys = silver_stops_df.filter("is_current").select("node", "cis")
    return (
        staging_stops_df.alias("s")
        .join(current_keys.alias("t"), ["node", "cis"], "left_anti")
        .select(
            *STOPS_SILVER_COLS,
            F.col("source_timestamp").alias("valid_from"),
            F.lit(None).cast("timestamp").alias("valid_to"),
            F.lit(True).alias("is_current"),
        )
    )

In [0]:
staging = prepare_stops_staging(_make_bronze([
    _make_sg(node=1, cis=100),
    _make_sg(node=2, cis=200, name="New"),
]))

# Silver only has node=1 → node=2 is genuinely new
silver_df = spark.createDataFrame([Row(node=1, cis=100, is_current=True)])
new_recs = build_new_stop_records(staging, silver_df)

test_eq(new_recs.count(), 1)
test_eq(new_recs.collect()[0]["node"], 2)
test_eq(new_recs.collect()[0]["is_current"], True)
test_eq(new_recs.collect()[0]["valid_to"], None)

print("All build_new_stop_records tests passed!")

All build_new_stop_records tests passed!


In [0]:
#| export

def apply_stops_scd2(spark: SparkSession, bronze_stops_table: str, silver_stops_table: str) -> tuple[int, int]:
    """Run the full SCD2 merge: prepare staging from bronze, expire changed rows, insert new + changed."""
    staging_stops_df = prepare_stops_staging(spark.table(bronze_stops_table))
    silver = DeltaTable.forName(spark, silver_stops_table)
    now = datetime.now(UTC)

    # Step 1: Expire changed records
    to_expire = find_stops_to_expire(staging_stops_df, silver.toDF())
    expired_count = to_expire.count()

    if expired_count > 0:
        silver.alias("t").merge(
            to_expire.alias("s"),
            "t.node = s.node AND t.cis = s.cis AND t.is_current = TRUE",
        ).whenMatchedUpdate(set={
            "is_current": F.lit(False),
            "valid_to": F.lit(now),
        }).execute()

    # Step 2: Insert new + changed records (reads silver AFTER expire)
    new_records = build_new_stop_records(staging_stops_df, silver.toDF())
    inserted_count = new_records.count()

    if inserted_count > 0:
        new_records.write.mode("append").saveAsTable(silver_stops_table)

    return expired_count, inserted_count

In [0]:
# apply_stops_scd2 orchestrates the full SCD2 merge against real Delta tables.
# TODO: add end-to-end integration test with temporary bronze + silver tables.
print("apply_stops_scd2: integration function — add integration test with real tables")

apply_stops_scd2: integration function — add integration test with real tables


In [0]:
nbdev.export.nb_export('03_cleaning_functions.ipynb', './exports')